# 🛡️ End-to-End Spam & Cyber Threat Detection Experiment

This notebook demonstrates the complete machine learning lifecycle:
1. **Dataset Loading & Exploration** (SMS Spam Collection)
2. **Text Preprocessing** (Lowercasing, Tokenization, Stopword Removal, Stemming, Feature Extraction)
3. **Stratified Splitting** (70% Train, 15% Validation, 15% Test without data leakage)
4. **Comparative Feature Extraction**: Bag-of-Words (BoW) vs. TF-IDF
5. **Hyperparameter Tuning & Cross-Validation** (GridSearchCV across 5 ML architectures)
6. **Model Evaluation & Leaderboard Comparison** (Accuracy, Precision, Recall, F1, ROC-AUC)
7. **Token Explainability & Probability Calibration**

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root in python path
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.dataset import load_dataset
from src.preprocess import Preprocessor, stratified_split_70_15_15, get_class_weights
from src.feature_extraction import ComparativeFeatureExtractor, tune_vectorizer_hyperparameters
from src.train_ml import get_model_search_spaces, evaluate_model

print("Environment initialized successfully. Random state fixed to 42.")

## 1. Load Dataset & Inspect Distribution

In [ ]:
df = load_dataset("sms")
print(f"Total dataset shape: {df.shape}")
print("Class balance:")
print(df["label"].value_counts(normalize=True) * 100)
df.head()

## 2. Text Preprocessing & Stratified 70/15/15 Split
Strictly avoid data leakage: we clean text, perform a stratified 70/15/15 split, and fit vectorizers **only** on the training partition.

In [ ]:
preprocessor = Preprocessor()
df["clean_text"] = preprocessor.preprocess_series(df["text"])

# Extract engineered extra features
extra_features_sample = preprocessor.extract_features_df(df["text"].iloc[:5])
print("Extracted Lexical / Numeric Features:")
display(extra_features_sample)

# Stratified 70/15/15 split
train_df, val_df, test_df = stratified_split_70_15_15(df, target_col="target", random_state=42)
print(f"Train samples: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val samples:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test samples:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

## 3. Comparative Feature Extraction: BoW vs. TF-IDF
Fit vectorizers strictly on `train_df['clean_text']`.

In [ ]:
extractor = ComparativeFeatureExtractor(max_features=5000, ngram_range=(1, 2))
X_train_bow, X_train_tfidf = extractor.fit_transform_train(train_df["clean_text"].values)
X_test_bow, X_test_tfidf = extractor.transform_eval(test_df["clean_text"].values)

print(f"X_train_bow shape:   {X_train_bow.shape}")
print(f"X_train_tfidf shape: {X_train_tfidf.shape}")
diag = extractor.get_vocabulary_diagnostics()
print(f"Vocabulary size: {diag['tfidf_vocab_size']}")
print("Sample bigrams extracted:", diag["sample_bigrams"][:5])

## 4. Model Training & Evaluation Benchmark
Compare models across both representations on the held-out test split.

In [ ]:
from src.evaluate import run_full_evaluation
# Run comprehensive benchmark and generate CSV table and ROC charts
run_full_evaluation(dataset_name="sms")

## 5. Visualizing Benchmark Comparison & ROC Curves

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(PROJECT_ROOT / "reports" / "bow_vs_tfidf_comparison_bars.png")))
display(Image(filename=str(PROJECT_ROOT / "reports" / "roc_curves_comparison.png")))

## 6. Live Inference & Explainability Demo

In [ ]:
from src.predict import SpamPredictor

predictor = SpamPredictor(model_name="svm_rbf", representation="tfidf")
sample_msg = "WINNER! You have won £1,000 cash guarantee! Call 09061701461 to claim right now."
result = predictor.predict(sample_msg)

print(f"Prediction:  {result['prediction']}")
print(f"Spam Probability: {result['spam_probability_pct']}%")
print(f"Confidence:  {result['confidence_pct']}%")
print("Top Token Attributions:")
for kw in result["top_influencing_words"]:
    print(f"  • {kw['word']}: {kw['score']} ({kw['sentiment']})")